# Task 1: Data Analysis, Preprocessing, and EDA

This notebook guides through the entire Task 1 pipeline. We will import modular functions from the `scripts` directory to keep this workspace clean and focused on analysis and visualization.

### Step 1: Setup and Imports

First, we import the necessary libraries for data manipulation (`pandas`), visualization (`seaborn`, `matplotlib`), and our custom pipeline functions.

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import os

# Add the project root to the Python path to allow for module imports
# This allows us to import from the 'scripts' directory
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Import our custom modules
from scripts import config
from scripts import task1_pipeline as pipe

# Set visualization style for all plots
sns.set_style('whitegrid')

### Step 2: Load the Datasets

We load the e-commerce fraud data and the IP address mapping data using our custom loader function from the `task1_pipeline` module.

In [6]:
fraud_df = pipe.load_data(config.FRAUD_DATA_PATH)
ip_df = pipe.load_data(config.IP_DATA_PATH)

# Display the first few rows of each dataframe to verify they loaded correctly
if fraud_df is not None:
    print("--- Fraud Data Head ---")
    display(fraud_df.head())
if ip_df is not None:
    print("\n--- IP Address to Country Data Head ---")
    display(ip_df.head())

🔄 Loading data from: d:\matos\tenx 10academy\week 8\Improved detection of fraud cases for e-commerce and bank transactions\data\raw\Fraud_Data.csv
✅ Data loaded successfully.
   Shape of the dataframe: (151112, 11)
🔄 Loading data from: d:\matos\tenx 10academy\week 8\Improved detection of fraud cases for e-commerce and bank transactions\data\raw\IpAddress_to_Country.csv
✅ Data loaded successfully.
   Shape of the dataframe: (138846, 3)
--- Fraud Data Head ---


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0



--- IP Address to Country Data Head ---


,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


### Step 3: Data Cleaning and Type Correction

Here, we'll handle duplicate entries and ensure time-related columns are in the correct `datetime` format, which is crucial for time-based feature engineering.

In [7]:
fraud_df_cleaned = pipe.clean_and_prepare_data(fraud_df)

# Verify the data types, especially for the time columns
print("\n--- Data Info After Cleaning ---")
fraud_df_cleaned.info()


--- Starting Data Cleaning and Preparation ---
Initial number of rows: 151112
Number of rows after dropping duplicates: 151112
🔄 Converting time columns to datetime objects...
✅ Time columns converted successfully.

--- Data Info After Cleaning ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   user_id         151112 non-null  int64         
 1   signup_time     151112 non-null  datetime64[ns]
 2   purchase_time   151112 non-null  datetime64[ns]
 3   purchase_value  151112 non-null  int64         
 4   device_id       151112 non-null  object        
 5   source          151112 non-null  object        
 6   browser         151112 non-null  object        
 7   sex             151112 non-null  object        
 8   age             151112 non-null  int64         
 9   ip_address      151112 non-null  float64       
 10

### Step 4: Geolocation Analysis - Merging Datasets

We enrich our transaction data by adding a `country` feature. This is done by converting the transaction IP address to an integer and finding which country's IP range it falls into.

In [8]:
merged_df = pipe.merge_with_ip_data(fraud_df_cleaned, ip_df)

# Display the head to see the new 'country' column
print("\n--- Data Head After Merging with Country ---")
display(merged_df.head())


--- Merging with Geolocation Data ---
🔄 Converting float 'ip_address' column to integer format...
✅ 'ip_address' converted to integer.
🔄 Merging dataframes based on IP range. This may take a moment...
✅ Merge complete. Found 182 unique countries.
   Transactions with 'Unknown' country: 21966

--- Data Head After Merging with Country ---


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,country
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0,Japan
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0,United States
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1,United States
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0,Unknown
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0,United States


### Step 5: Feature Engineering

Now we create new features that might provide strong signals for our fraud detection model. This includes time differences, time-based features, and transaction frequencies.

In [9]:
featured_df = pipe.create_features(merged_df)

# Display the head to see all the new features
print("\n--- Data Head After Feature Engineering ---")
display(featured_df[['user_id', 'device_id', 'time_since_signup_seconds', 'purchase_hour_of_day', 'device_id_count', 'class']].head())


--- Starting Feature Engineering ---
✅ Feature 'time_since_signup_seconds' created.
✅ Features 'purchase_hour_of_day' and 'purchase_day_of_week' created.
✅ Features 'device_id_count' and 'user_id_count' created.

--- Data Head After Feature Engineering ---


,user_id,device_id,time_since_signup_seconds,purchase_hour_of_day,device_id_count,class
0,22058,QVPSPJUOCKZAR,4506682.0,2,1,0
1,333320,EOGFQPIZPYXFZ,17944.0,1,1,0
2,1359,YSSKYOSJHPPLJ,1.0,18,12,1
3,150084,ATGTXKYKUDUQN,492085.0,13,1,0
4,221365,NAUITBZFJKHWW,4361461.0,18,1,0


### Step 6: Exploratory Data Analysis (EDA)

With our cleaned and feature-enriched data, we can now perform EDA to find insights.

#### Critical Challenge: Class Imbalance

As stated in the project description, the dataset is highly imbalanced. Let's visualize this to understand the scale of the problem.

In [ ]:
print("Class Distribution:")
class_dist_percent = featured_df['class'].value_counts(normalize=True) * 100
print(f"Non-Fraud (0): {class_dist_percent[0]:.2f}%")
print(f"Fraud (1):     {class_dist_percent[1]:.2f}%")

plt.figure(figsize=(8, 5))
sns.countplot(x='class', data=featured_df)
plt.title('Class Distribution (0: Non-Fraud, 1: Fraud)', fontsize=16)
plt.xlabel('Class', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Save the plot to the outputs/images directory
plt.savefig(os.path.join(config.IMAGE_DIR, 'class_distribution.png'))
plt.show()

#### Bivariate Analysis

Let's explore the relationship between our new features and the target variable (`class`). This can reveal which features are most likely to be predictive of fraud.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Feature Comparison for Fraud vs. Non-Fraud', fontsize=18)

# 1. Time Since Signup vs. Class
sns.boxplot(x='class', y='time_since_signup_seconds', data=featured_df, ax=axes[0])
axes[0].set_title('Time Since Signup', fontsize=14)
axes[0].set_yscale('log') # Use log scale due to wide distribution of values
axes[0].set_ylabel('Time Since Signup (seconds, log scale)', fontsize=12)
axes[0].set_xlabel('Class', fontsize=12)

# 2. Purchase Value vs. Class
sns.boxplot(x='class', y='purchase_value', data=featured_df, ax=axes[1])
axes[1].set_title('Purchase Value', fontsize=14)
axes[1].set_ylabel('Purchase Value ($)', fontsize=12)
axes[1].set_xlabel('Class', fontsize=12)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(config.IMAGE_DIR, 'bivariate_analysis.png'))
plt.show()

**Insight:** The plots strongly suggest that fraudulent transactions (`class=1`) tend to happen very shortly after signup, as shown by the much lower median `time_since_signup_seconds`. This is a powerful indicator. Purchase value, on the other hand, does not show as clear a separation between the two classes, although the distribution for fraudulent transactions appears slightly tighter.

### Step 7: Save Processed Data

Finally, we save the fully processed dataframe to the `data/processed` directory. This file will be the input for Task 2 (Model Building).

In [ ]:
try:
    featured_df.to_csv(config.PROCESSED_FRAUD_DATA_PATH, index=False)
    print(f"\n✅ Successfully saved the final processed data to: {config.PROCESSED_FRAUD_DATA_PATH}")
    print("\n--- Final DataFrame Info ---")
    featured_df.info()
except Exception as e:
    print(f"❌ Failed to save the processed data. Error: {e}")